<a href="https://colab.research.google.com/github/minyi-k03/Large-Language-Model-LLM-/blob/Fine-Tuning/Llama2_fine_tuning_korquad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Llama 2를 KorQuad 데이터셋에 맞게 fine-tuning 하기
### References:
1.   https://www.youtube.com/watch?v=LslC2nKEEGU
2.   https://colab.research.google.com/drive/1JDnGJbxT8fSqwnXY8J-XFo73AtiSuQMe?usp=sharing

In [ ]:
!nvidia-smi

## AutoTrain Advanced
https://github.com/huggingface/autotrain-advanced

In [ ]:
import os

print("🛑 설치 시작. 절대 끄지 마세요.")

# 1. 기존 찌꺼기 제거
!pip uninstall -y transformers autotrain-advanced peft triton bitsandbytes accelerate > /dev/null 2>&1

# 2. 필수 라이브러리 설치 (최신 버전 사용)
# autotrain-advanced는 최신 transformers에 의존하므로 전부 최신으로 맞춥니다.
!pip install -U autotrain-advanced
!pip install -U bitsandbytes
!pip install -U transformers peft accelerate torch torchvision torchaudio

print("✅ 설치 완료. 상단 메뉴 [런타임] -> [세션 다시 시작]을 누르세요.")

## PyTorch 업데이트

In [ ]:
!autotrain setup --update-torch

## Korquad 데이터셋에 맞게 Llama2 Fine-Tuning

- autotrain 설정값: https://github.com/huggingface/autotrain-advanced/blob/f1367b590dfc53d240e9684779991da540590386/src/autotrain/cli/run_llm.py#L21 (**과거 버전[0.6.35]**)
- autotrain 설정값: https://github.com/huggingface/autotrain-advanced/blob/main/src/autotrain/cli/run_llm.py#L17 (**최신 버전[0.6.80]**)


In [ ]:
# 1. 기존 폴더 정리
!rm -rf llama2-korquad-finetuning-da

# 2. 학습 시작 (GPU 경로 강제 주입 모드)
# 아래 명령어 한 줄이 핵심입니다. 절대 수정하지 마세요.
!LD_LIBRARY_PATH=/usr/lib64-nvidia:/usr/local/cuda/lib64:$LD_LIBRARY_PATH \
 autotrain llm --train \
    --project-name "llama2-korquad-finetuning-da" \
    --model "TinyPixel/Llama-2-7B-bf16-sharded" \
    --data-path "korquad_prompt_da" \
    --text-column "text" \
    --peft \
    --quantization "int4" \
    --lr 1e-4 \
    --batch-size 8 \
    --epochs 10 \
    --trainer sft \
    --model_max_length 256

# 학습결과 zip 파일로 압축후 다운로드

In [ ]:
import zipfile
import shutil
from google.colab import files

# 압축할 폴더 이름
#folder_name = "llama2-korquad-finetuning"    # Data Augmentation 적용 x
folder_name = "llama2-korquad-finetuning-da"  # Data Augmentation 적용 o

# 생성될 ZIP 파일 이름
#zip_file_name = "llama2-korquad-finetuning.zip"  # Data Augmentation 적용 x
zip_file_name = "llama2-korquad-finetuning-da.zip" # Data Augmentation 적용 o

# 폴더를 ZIP 파일로 압축
shutil.make_archive(zip_file_name[:-4], 'zip', folder_name)

# ZIP 파일을 로컬로 다운로드
files.download(zip_file_name)

# Llama-2 Fine-Tuning 모델 성능 테스트

In [ ]:
import torch
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. 경로 설정 (학습 결과 폴더 이름)
peft_model_id = "llama2-korquad-finetuning-da"

# 2. 기본 모델(Llama-2) + 토크나이저 불러오기
config = PeftConfig.from_pretrained(peft_model_id)
base_model = AutoModelForCausalLM.from_pretrained(
    "TinyPixel/Llama-2-7B-bf16-sharded",
    return_dict=True,
    torch_dtype=torch.float16,
    device_map='auto'
)
tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path)

# 3. 우리가 만든 학습 결과(Adapter) 합체!
model = PeftModel.from_pretrained(base_model, peft_model_id)

# 4. 질문 던져보기 (테스트)
def ask_model(question):
    # 프롬프트 형식은 학습시킬 때 썼던 형식과 맞춰주는 게 좋습니다.
    prompt = f"질문: {question}\n답변:"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # 생성
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            max_new_tokens=50,  # 답변 길이
            do_sample=True,     # 창의적 생성
            top_p=0.9,
            temperature=0.7
        )

    # 결과 해독
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

# 실행!
print("🤖 모델 로딩 완료! 테스트를 시작합니다.")
ask_model("대한민국 수도는?")